# Calibration analysis

**Purpose.** Reliability curves and expected calibration error (ECE), per
arm × outcome, and per-outcome temperature scaling fit on the validation
predictions and applied unchanged to test. Reads only
`outputs/<arm>/test_predictions.csv` — no re-training required.

Sections:

1. Load per-arm test predictions.
2. Per-outcome ECE across every arm (support-weighted summary).
3. Reliability curves for the KB and KB+QA arms.
4. Temperature scaling — fit on validation, evaluate on test, report the
   ECE delta.
5. Export tables + figures to `outputs/figures/calibration/`.

This notebook exists because calibration is post-hoc analysis on the
ladder's own predictions, unrelated to the training loop. Keeping it
outside `benchmark.ipynb` avoids re-running any expensive cell when the
only thing that changes is the plotting.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kineret.config import paths
from kineret.benchmark import ARMS, arm_label
from kineret.evaluation import (
    load_predictions, outcome_names_from,
    reliability_data, expected_calibration_error,
    fit_temperature, apply_temperature,
)

OUT_ROOT = paths.OUTPUT_ROOT
FIG_DIR  = os.path.join(OUT_ROOT, "figures", "calibration")
os.makedirs(FIG_DIR, exist_ok=True)

## 1 — Load predictions

One `test_predictions.csv` per arm. The 7-arm ladder is fully loaded
when this cell finishes.

In [ ]:
arm_preds = {}
for arm_key, spec in ARMS.items():
    run_dir = os.path.join(OUT_ROOT, spec["run_dir"])
    pred_path = os.path.join(run_dir, "test_predictions.csv")
    if not os.path.exists(pred_path):
        print(f"[skip] {arm_key}: no predictions at {pred_path}")
        continue
    arm_preds[arm_key] = load_predictions(run_dir)
    print(f"[ok]   {arm_label(arm_key)}: {len(arm_preds[arm_key])} patients")

OUTCOMES = outcome_names_from(next(iter(arm_preds.values())))
print(f"\nOutcomes ({len(OUTCOMES)}): {OUTCOMES}")

## 2 — Per-outcome ECE (uniform bins, matching the ECE convention)

In [ ]:
rows = []
for arm_key, preds in arm_preds.items():
    for o in OUTCOMES:
        y = preds[f"label_{o}"].to_numpy(dtype=float)
        p = preds[f"prob_{o}"].to_numpy(dtype=float)
        rows.append({
            "arm": arm_label(arm_key),
            "outcome": o,
            "n_pos": int(np.nansum(y == 1)),
            "ECE_15": expected_calibration_error(y, p, n_bins=15),
        })

ece_df = pd.DataFrame(rows)

# Support-weighted ECE per arm.
def _weighted(g):
    w = g["n_pos"].to_numpy(dtype=float)
    if w.sum() == 0:
        return float("nan")
    return float(np.average(g["ECE_15"], weights=w))

per_arm = ece_df.groupby("arm").apply(_weighted).rename("ECE_weighted")

ece_df.to_csv(os.path.join(FIG_DIR, "per_outcome_ece.csv"), index=False)
per_arm.to_csv(os.path.join(FIG_DIR, "per_arm_ece.csv"))

per_arm.round(4)

## 3 — Reliability curves (KB and KB+QA arms)

One panel per outcome. Bins are quantile-defined so each bin holds
roughly the same number of test patients, which matters at low prevalence
where uniform bins would leave the high-probability end empty.

In [ ]:
arms_to_plot = ["intervene_kb", "intervene_kb_qa"]
arms_to_plot = [a for a in arms_to_plot if a in arm_preds]

n_out = len(OUTCOMES)
cols = 4
rows = (n_out + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.2 * rows),
                          squeeze=False)
for ax, outcome in zip(axes.flat, OUTCOMES):
    ax.plot([0, 1], [0, 1], "k--", lw=0.6, alpha=0.6)
    for arm_key in arms_to_plot:
        preds = arm_preds[arm_key]
        y = preds[f"label_{outcome}"].to_numpy(dtype=float)
        p = preds[f"prob_{outcome}"].to_numpy(dtype=float)
        rel = reliability_data(y, p, n_bins=10, strategy="quantile")
        ax.plot(rel["mean_pred"], rel["mean_obs"], marker="o",
                label=arm_label(arm_key), ms=4, lw=1.2)
    ax.set_title(outcome, fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("predicted"); ax.set_ylabel("observed")
for ax in axes.flat[n_out:]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "reliability_kb.png"), dpi=200)
fig.savefig(os.path.join(FIG_DIR, "reliability_kb.pdf"))
fig

## 4 — Temperature scaling

Fit a per-outcome temperature $T$ on **validation** predictions and
apply it unchanged to **test**. This is the standard
post-hoc calibration correction (Guo et al. 2017); it changes only the
probability calibration, not the ranking, so AUROC / AUPRC are
unaffected.

This cell requires that validation predictions exist for the arm
(saved as `val_predictions.csv` alongside `test_predictions.csv`). If
your training loop did not emit them, either:

- re-run the arm with `save_val_predictions=True` (a one-flag change in
  each arm's `train.py`), or
- fit $T$ on a split-off of test and report both raw and scaled ECE
  transparently (documented in the code below).

The pre-shipped 7-arm run does not currently emit validation
predictions, so the cell below falls back to test-fit + test-apply and
reports both numbers with an explicit caveat.

In [ ]:
def try_load_val(arm_key):
    spec = ARMS[arm_key]
    run_dir = os.path.join(OUT_ROOT, spec["run_dir"])
    val = os.path.join(run_dir, "val_predictions.csv")
    return pd.read_csv(val) if os.path.exists(val) else None

rows = []
for arm_key in arms_to_plot:
    val_df = try_load_val(arm_key)
    used_split = "validation" if val_df is not None else "test (self-fit)"
    fit_source = val_df if val_df is not None else arm_preds[arm_key]

    for o in OUTCOMES:
        y_fit = fit_source[f"label_{o}"].to_numpy(dtype=float)
        p_fit = fit_source[f"prob_{o}"].to_numpy(dtype=float)
        T = fit_temperature(y_fit, p_fit, is_prob=True)

        y_test = arm_preds[arm_key][f"label_{o}"].to_numpy(dtype=float)
        p_test = arm_preds[arm_key][f"prob_{o}"].to_numpy(dtype=float)
        ece_raw  = expected_calibration_error(y_test, p_test)
        ece_temp = expected_calibration_error(
            y_test, apply_temperature(p_test, T)
        )
        rows.append({
            "arm":       arm_label(arm_key),
            "outcome":   o,
            "T":         round(T, 3),
            "ECE_raw":   round(ece_raw, 4),
            "ECE_temp":  round(ece_temp, 4),
            "fit_split": used_split,
        })

temp_df = pd.DataFrame(rows)
temp_df.to_csv(os.path.join(FIG_DIR, "temperature_scaling.csv"), index=False)
temp_df

## 5 — Exported artefacts

```
outputs/figures/calibration/
├── per_outcome_ece.csv         # arm × outcome ECE (uniform 15-bin)
├── per_arm_ece.csv             # weighted per-arm ECE
├── reliability_kb.png / .pdf   # 8-panel reliability, KB and KB+QA
└── temperature_scaling.csv     # per-outcome T + ECE before / after
```

These are the inputs the AIIM discussion's calibration paragraph reads.